## Machine Learning – Prédiction du prix des billets d’avion ##

### Preprocessing : ##

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [41]:
df = pd.read_csv('Clean_Dataset.csv')
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


##### 1.  Suppression des colonnes inutiles ##

In [42]:
df = df.drop(columns= ['Unnamed: 0', 'flight'])
df.head()

,airline,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,SpiceJet,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,AirAsia,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


##### 2.  Encodage ordinal

In [43]:
stops_order = ['zero', 'one', 'two_or_more']
class_order = ['Economy', 'Business']

stops_dict = {
    'zero': 0,
    'one': 1,
    'two_or_more': 2 }

class_dict = {
    'Economy': 0,
    'Business': 1 }


df['stops_encoded'] = df['stops'].map(stops_dict)
df['class_order'] = df['class'].map(class_dict)
df.head()

,airline,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,stops_encoded,class_order
0,SpiceJet,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,0,0
1,SpiceJet,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,0,0
2,AirAsia,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,0,0
3,Vistara,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,0,0
4,Vistara,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,0,0


##### 3.  Encodage nominal

In [ ]:
# Si on veut utiliser get_dummies
# nominal_features = ['airline', 'source_city', 'departure_time', 'arrival_time', 'destination_city']
# df = pd.get_dummies(df, columns=nominal_features, dtype=int)
# df.head()

In [ ]:

nominal_features = ['airline', 'source_city', 'departure_time', 'arrival_time', 'destination_city']
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=int)
encoded = ohe.fit_transform(df[nominal_features])

# Récupérer les noms de colonnes
encoded_cols = ohe.get_feature_names_out(nominal_features)

df = df.drop(columns=nominal_features)
df = pd.concat([df, pd.DataFrame(encoded, columns=encoded_cols, index=df.index)], axis=1)

df.head()

,stops,class,duration,days_left,price,stops_encoded,class_order,airline_AirAsia,airline_Air_India,airline_GO_FIRST,...,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night,destination_city_Bangalore,destination_city_Chennai,destination_city_Delhi,destination_city_Hyderabad,destination_city_Kolkata,destination_city_Mumbai
0,zero,Economy,2.17,1,5953,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1,zero,Economy,2.33,1,5953,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
2,zero,Economy,2.17,1,5956,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
3,zero,Economy,2.25,1,5955,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,zero,Economy,2.33,1,5955,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1


##### 4.  Scaling & Splitting 

In [45]:

X = df.drop(columns=['price'])
y = df['price']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train[['duration', 'days_left']] = scaler.fit_transform(
    X_train[['duration', 'days_left']]
)

X_test[['duration', 'days_left']] = scaler.transform(
    X_test[['duration', 'days_left']]
)

### Modélisation : ##